# MedStream - Interactive Clinical Dashboard

An interactive notebook dashboard for clinical monitoring and real-time diabetes risk assessment:
- Queries key metrics (KPIs) from the DuckDB analytical warehouse (`patients_analytics`).
- Visualizes age, glucose, and HbA1c distributions by diagnosis.
- Provides interactive input sliders to predict diabetes probability using the trained Random Forest model.

### 1. Load Warehouse Metrics

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect('medstream_warehouse.duckdb')
df = con.execute("SELECT * FROM patients_analytics").df()
con.close()

total_pts = len(df)
diabetic_pts = (df['diabetes'] == 1).sum()
diabetic_pct = (diabetic_pts / total_pts) * 100
avg_glucose = df['blood_glucose_level'].mean()
avg_hba1c = df['HbA1c_level'].mean()

print("=" * 50)
print("       MEDSTREAM WAREHOUSE OVERVIEW")
print("=" * 50)
print(f"Total Patient Records:  {total_pts:,}")
print(f"Diagnosed Patients:     {diabetic_pts:,} ({diabetic_pct:.2f}%)")
print(f"Mean Blood Glucose:     {avg_glucose:.1f} mg/dL")
print(f"Mean HbA1c Level:       {avg_hba1c:.2f}")
print("=" * 50)

### 2. Clinical KPI Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Age by diagnosis
df.boxplot(column='age', by='diabetes', ax=axes[0])
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[0].set_ylabel('Age')

# 2. Glucose by diagnosis
df.boxplot(column='blood_glucose_level', by='diabetes', ax=axes[1])
axes[1].set_title('Blood Glucose Distribution')
axes[1].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[1].set_ylabel('Glucose (mg/dL)')

# 3. HbA1c by diagnosis
df.boxplot(column='HbA1c_level', by='diabetes', ax=axes[2])
axes[2].set_title('HbA1c Level Distribution')
axes[2].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[2].set_ylabel('HbA1c')

plt.suptitle('')
plt.tight_layout()
plt.show()

### 3. Load Trained Model for Real-Time Inference

In [ ]:
import joblib

model = joblib.load('models/diabetes_model.pkl')
feature_cols = joblib.load('models/model_columns.pkl')
print(f"Loaded trained model with {len(feature_cols)} input features")

### 4. Interactive Patient Risk Predictor
Adjust the sliders below to test risk scores for hypothetical or actual patients.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# input controls
w_gender = widgets.Dropdown(options=['Female', 'Male'], value='Female', description='Gender:')
w_age = widgets.IntSlider(value=50, min=1, max=100, step=1, description='Age:')
w_hyper = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=0, description='Hypertension:')
w_heart = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=0, description='Heart Dis.:')
w_smoke = widgets.Dropdown(options=['never', 'former', 'current', 'No Info'], value='never', description='Smoking:')
w_bmi = widgets.FloatSlider(value=27.5, min=10.0, max=60.0, step=0.1, description='BMI:')
w_hba1c = widgets.FloatSlider(value=5.8, min=3.5, max=10.0, step=0.1, description='HbA1c:')
w_glucose = widgets.IntSlider(value=130, min=60, max=300, step=1, description='Glucose:')
btn_predict = widgets.Button(description='Run Assessment', button_style='primary')
out = widgets.Output()

def on_predict_clicked(b):
    with out:
        out.clear_output()
        # build patient row
        patient_dict = {
            'age': w_age.value,
            'hypertension': w_hyper.value,
            'heart_disease': w_heart.value,
            'bmi': w_bmi.value,
            'HbA1c_level': w_hba1c.value,
            'blood_glucose_level': w_glucose.value,
            'gender_code': 1 if w_gender.value == 'Male' else 0,
            'gender_Male': 1 if w_gender.value == 'Male' else 0,
            'smoking_history_former': 1 if w_smoke.value == 'former' else 0,
            'smoking_history_never': 1 if w_smoke.value == 'never' else 0,
            'smoking_history_No Info': 1 if w_smoke.value == 'No Info' else 0
        }
        
        patient_df = pd.DataFrame([patient_dict])
        # align columns with training schema
        for col in feature_cols:
            if col not in patient_df.columns:
                patient_df[col] = 0
        patient_df = patient_df[feature_cols]
        
        # model inference
        prob = model.predict_proba(patient_df)[0][1] * 100
        pred = model.predict(patient_df)[0]
        
        if pred == 1:
            badge = f"<div style='background-color:#ffebee; padding:12px; border-left:6px solid #d32f2f; border-radius:4px;'>" \
                    f"<h3 style='color:#c62828; margin:0;'>HIGH RISK DETECTED ({prob:.1f}%)</h3>" \
                    f"<p style='margin:4px 0 0 0;'>Patient shows significant clinical indicators of diabetes. Clinical follow-up recommended.</p></div>"
        else:
            badge = f"<div style='background-color:#e8f5e9; padding:12px; border-left:6px solid #2e7d32; border-radius:4px;'>" \
                    f"<h3 style='color:#1b5e20; margin:0;'>LOW RISK ({prob:.1f}%)</h3>" \
                    f"<p style='margin:4px 0 0 0;'>Patient indicators fall within expected non-diabetic ranges.</p></div>"
        display(HTML(badge))

btn_predict.on_click(on_predict_clicked)

panel = widgets.VBox([
    widgets.HBox([w_gender, w_age]),
    widgets.HBox([w_hyper, w_heart]),
    widgets.HBox([w_bmi, w_smoke]),
    widgets.HBox([w_hba1c, w_glucose]),
    widgets.Box([btn_predict], layout=widgets.Layout(margin='10px 0')),
    out
])

display(panel)